# WH-M01: Medallion Sales Pipeline

| What to expect | Value |
| --- | --- |
| Scenario | Retail sales |
| Difficulty | Medium |
| Delivery scope | System flow across six stages |
| Expected effort | Multiple sessions |
| Deliverable | Rerunnable medallion warehouse with validation evidence |
| Primary interfaces | Python, pandas, SQL, PostgreSQL |
| Prerequisites | Basic Python, pandas DataFrames, and SQL queries |
| Tooling | `python`, `pandas`, `sql`, `postgresql`, `sqlalchemy` |
| Techniques | `raw-ingestion`, `data-profiling`, `data-quality`, `star-schema`, `scd1`, `scd2`, `reconciliation` |
| Database | `week3_medallion_lab` |

This lab will build a complete Bronze → Silver → Gold warehouse from a new deterministic source dataset. The notebook is the learning interface; reusable behavior lives in linked Python, SQL, script, and data artifacts.

Most inspection, profiling, cleaning, and reconciliation work will use staged pandas DataFrames. SQL will define durable PostgreSQL schemas, constraints, loads, and analytical queries; notebook cells will run that SQL through SQLAlchemy so the complete learning flow stays here.

## Start here

1. Read the pipeline map and Stage 0 checkpoints below.
2. Read the [lab commands](../../../scripts/interactive-data-engineering-labs/retail-sales/wh-m01-medallion-sales-pipeline/README.md).
3. Run Checkpoints 0.1 and 0.2 using the repository `.venv` kernel.
4. Review the cleanup choices before creating any data.

The [SQL directory](../../../sql/interactive-data-engineering-labs/retail-sales/wh-m01-medallion-sales-pipeline/README.md) and [source-data directory](../../../data/samples/interactive-data-engineering-labs/retail-sales/wh-m01-medallion-sales-pipeline/README.md) will gain their executable artifacts as you reach each stage.


## Pipeline map

```text
Generated source data
        ↓
Bronze: preserve raw data
        ↓
Profile and clean with pandas
        ├── accepted rows → Silver
        └── invalid rows  → rejection table
                                ↓
Silver: trusted typed records
        ↓
Resolve dimension keys
        ↓
Gold: star schema
        ↓
SQL analysis and visualization
```

The lab is separate from the Week 3 homework. It must not query, reset, or modify `week3_hw`, `bigdata`, the homework SQL, or the sample-warehouse CSV files.

Repeatability contract: the generated source delivery uses fixed inputs and a checksum manifest, Bronze adds stable source-lineage fields while preserving source values, each database stage has a bounded reset point, and every stage ends with a **Check my work** validation before the next layer is trusted.


## Stage 0: Environment and scaffold

### Checkpoint 0.1: Verify the project environment

<details>
<summary>Open checkpoint instructions and environment hint</summary>

Run the next cell. It locates the repository, imports the reusable lab helper, and reports installed package versions. It does not create data, schemas, or tables. Acceptance evidence is a dictionary containing all six required packages.

**Hint:** If an import fails, run the installation command in the lab README from the repository root, restart the kernel, and rerun this cell.

**Job connection:** Reproducible pipelines declare their runtime dependencies. Verifying the environment before touching persisted state separates setup failures from data or transformation failures.

</details>


In [ ]:
from importlib.metadata import version
from pathlib import Path
import sys

current_path = Path.cwd().resolve()
project_root = next(
    candidate
    for candidate in (current_path, *current_path.parents)
    if (candidate / "pyproject.toml").is_file()
    and (candidate / "AGENTS.md").is_file()
)
source_root = project_root / "src"
if str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))

from big_data_example.labs.retail_sales.wh_m01_database import (
    LAB_DATABASE,
    POSTGRES_CONTAINER,
    verify_container_health,
    verify_lab_connection,
)

package_versions = {
    package: version(package)
    for package in (
        "ipykernel",
        "matplotlib",
        "pandas",
        "psycopg2-binary",
        "seaborn",
        "SQLAlchemy",
    )
}
package_versions


### Checkpoint 0.2: Verify PostgreSQL and the isolated database

<details>
<summary>Open checkpoint instructions, hint, and job connection</summary>

Run the next cell. The first check asks Docker for the existing repository container's health. The second uses SQLAlchemy and the ignored `infra/postgres/.env` configuration to connect specifically to `week3_medallion_lab`. Acceptance evidence is `healthy` plus a database result naming `week3_medallion_lab`; the result never contains the password.

**Hint:** The `database_name` value proves which PostgreSQL database received the connection. The container name, PostgreSQL role, and database name are different identities.

**Job connection:** Data jobs should prove their target before creating or replacing tables. A valid password and healthy server do not guarantee that a job selected the correct database.

</details>


In [ ]:
container_status = verify_container_health()
database_evidence = verify_lab_connection(project_root)

{
    "container": POSTGRES_CONTAINER,
    "container_status": container_status,
    **database_evidence,
}


### Checkpoint 0.3: Inspect PostgreSQL with `psql`

<details>
<summary>Open PostgreSQL commands, hint, and job connection</summary>

From a VS Code terminal at the repository root, start or confirm the repository service:

```bash
docker compose --env-file infra/postgres/.env -f infra/postgres/compose.yml up -d
docker compose --env-file infra/postgres/.env -f infra/postgres/compose.yml ps
```

Enter `psql` through the existing container:

```bash
docker exec -it big-data-example-postgres psql -U bigdata -d postgres
```

At the `postgres=#` prompt, inspect the server and switch to the lab database:

```text
\l
\c week3_medallion_lab
\conninfo
\dn
\dt bronze.*
\dt silver.*
\dt gold.*
\q
```

At the Stage 0 boundary, the three `\dt` commands should report no matching relations. Tables will appear as you complete their stages.

**Hint:** Backslash commands are `psql` meta-commands entered after the PostgreSQL prompt. Do not enter them in Bash, and do not add semicolons. `-U bigdata` selects the PostgreSQL role; `-d postgres` selects the initial database.

**Job connection:** Engineers use both application connections and native database tools. Independent catalog inspection helps confirm that a pipeline wrote the intended objects to the intended environment.

</details>


### Checkpoint 0.4: Understand the safety boundary

<details>
<summary>Open safety table, hint, and job connection</summary>

| Operation | Exact target | Does not target |
| --- | --- | --- |
| Gold reset | `gold` inside `week3_medallion_lab` | Source delivery, Bronze, Silver, other databases, container, volume |
| Silver reset | `gold` and `silver` inside `week3_medallion_lab` | Source delivery, Bronze, other databases, container, volume |
| All-layer reset | `gold`, `silver`, and `bronze` inside `week3_medallion_lab` | Source delivery, other databases, container, volume |
| Full lab removal | The `week3_medallion_lab` database after exact-name confirmation | `week3_hw`, `bigdata`, container, volume |
| Notebook restart | In-memory Python state only | PostgreSQL objects and persisted data |
| Clear notebook outputs | Displayed notebook results only | Python state and PostgreSQL state |

**Hint:** Reset from the earliest layer you want to repeat. Because Gold is derived from Silver, a Silver reset removes Gold first.

**Job connection:** Layer-aware resets preserve trusted upstream inputs while preventing stale downstream tables from disagreeing with rebuilt data. Production backfills and recovery procedures need the same dependency awareness.

</details>


## Cleanup and repeatable restart points

<details>
<summary>Open cleanup commands and preservation rules</summary>

Run cleanup commands from a VS Code terminal opened at the repository root. These commands drop schemas rather than deleting selected rows, which gives each rebuilt layer a clean structure and avoids mixing old and new definitions.

Repeat Gold while keeping Silver and Bronze:

```bash
./.venv/bin/python scripts/interactive-data-engineering-labs/retail-sales/wh-m01-medallion-sales-pipeline/reset-lab.py gold
```

Repeat Silver and Gold while keeping the generated source delivery and Bronze:

```bash
./.venv/bin/python scripts/interactive-data-engineering-labs/retail-sales/wh-m01-medallion-sales-pipeline/reset-lab.py silver
```

Repeat ingestion, Silver, and Gold while keeping only the generated source delivery:

```bash
./.venv/bin/python scripts/interactive-data-engineering-labs/retail-sales/wh-m01-medallion-sales-pipeline/reset-lab.py schemas
```

Completely remove the isolated lab database:

```bash
./.venv/bin/python scripts/interactive-data-engineering-labs/retail-sales/wh-m01-medallion-sales-pipeline/reset-lab.py database --confirm week3_medallion_lab
```

**Destructive for this lab:** the final command removes every object in `week3_medallion_lab`. None of these commands remove `week3_hw`, `bigdata`, the PostgreSQL container, the Docker volume, or the generated source delivery. Do not use `docker compose down --volumes` for this lab.

</details>


## Stage 1: Generated source data and Bronze

Source grain: one source record represents one product line reported for a transaction. The delivery intentionally violates uniqueness and other quality expectations so Bronze can preserve problems for later diagnosis.

Bronze adds `bronze_row_id`, `source_batch_id`, `source_file_name`, `source_row_number`, and `loaded_at`. The eleven source columns remain text and retain their exact source values.

### Checkpoint 1.1: Generate and verify the source delivery

<details>
<summary>Open checkpoint instructions, hint, and job connection</summary>

Run the next cell to generate or verify the source CSV and manifest. The generator uses seed `20260917`, refuses to silently overwrite a changed delivery, and verifies the exact expected bytes.

Input objects: deterministic Python records. Output objects: one CSV path, one manifest path, and a Python `dict` containing the 40-row integrity contract. No transaction row is cleaned or rejected.

**Hint:** If verification reports that a file differs, inspect the change first. Use `generate-source.py --force` only when you intentionally want to restore the known delivery.

**Job connection:** Bronze provides replayable evidence of what arrived. Keeping imperfect source values prevents a later cleaning-rule change from requiring the original system to resend data.

</details>


In [ ]:
from big_data_example.labs.retail_sales.wh_m01_source import (
    SOURCE_COLUMNS,
    verify_source_delivery,
    write_source_delivery,
)

source_csv_path, source_manifest_path = write_source_delivery()
source_manifest = verify_source_delivery()
source_manifest


### Checkpoint 1.2: Read and profile the source with pandas

<details>
<summary>Open pandas object-and-shape explanation, hint, and job connection</summary>

Input object: a CSV file with 40 data rows and 11 source columns. `pd.read_csv` creates a pandas `DataFrame` with shape `(40, 11)`. `dtype="string"` keeps every column source-oriented, while `keep_default_na=False` preserves missing CSV fields as empty strings instead of silently converting them to pandas missing values.

The profile operations return small DataFrames: `head(8)` returns 8 rows × 11 columns, the column profile returns 11 rows × 3 columns, and the duplicate filter returns the two rows sharing a transaction ID. All 40 input rows survive unchanged.

**Hint:** Compare `null_count` with `empty_string_count`. A zero null count does not mean the source has no missing values when empty strings are preserved intentionally.

**Job connection:** Raw-file readers make parsing choices before cleaning begins. Recording those choices prevents missing values and identifiers from changing meaning merely because a library inferred a type.

</details>


In [ ]:
import pandas as pd

source_df = pd.read_csv(
    source_csv_path,
    dtype="string",
    keep_default_na=False,
)
print(f"Object type: {type(source_df).__name__}")
print(f"Shape: {source_df.shape}")
display(source_df.head(8))

source_column_profile_df = pd.DataFrame(
    {
        "dtype": source_df.dtypes.astype(str),
        "null_count": source_df.isna().sum(),
        "empty_string_count": source_df.eq("").sum(),
    }
)
display(source_column_profile_df)

duplicate_transaction_df = source_df.loc[
    source_df["transaction_id"].duplicated(keep=False)
].copy()
print(f"Duplicate-ID subset shape: {duplicate_transaction_df.shape}")
display(duplicate_transaction_df)


### Checkpoint 1.3: Inspect the intentional quality cases

<details>
<summary>Open subset explanation, hint, and job connection</summary>

Input object: `source_df`, a `(40, 11)` pandas DataFrame. The manifest supplies one-based source row numbers for the deliberate examples. `iloc` selects those positions and returns a new `(17, 11)` DataFrame named `intentional_issue_df`. No source row changes; this is inspection only.

**Hint:** One row can demonstrate more than one issue, and the two customer-city rows are valid records needed later for SCD Type 2. Do not assume every row listed by the manifest should be rejected.

**Job connection:** A data-quality fixture should state why edge cases exist. This makes tests reviewable and prevents a future maintainer from 'fixing' deliberate bad data.

</details>


In [ ]:
intentional_issue_rows = sorted(
    {
        row_number
        for row_numbers in source_manifest["intentional_quality_cases"].values()
        for row_number in row_numbers
    }
)
intentional_issue_df = source_df.iloc[
    [row_number - 1 for row_number in intentional_issue_rows]
].copy()
print(f"Object type: {type(intentional_issue_df).__name__}")
print(f"Shape: {intentional_issue_df.shape}")
display(intentional_issue_df)


### Checkpoint 1.4: Create and load Bronze

<details>
<summary>Open loading instructions, hint, and job connection</summary>

Run the next cell. Versioned SQL creates `bronze.sales_transactions`; Python verifies the manifest, replaces only `retail-sales-batch-001`, and loads all 40 rows with stable lineage. Rerunning the cell replaces that batch rather than appending duplicates.

Input objects: 40 dictionaries read from the verified CSV. Output: 40 PostgreSQL Bronze rows with 16 columns—five Bronze metadata columns plus the eleven unchanged source columns. No rows are rejected in Bronze.

**Hint:** Bronze uses text for source fields because an invalid date or nonnumeric price must still be loadable. Typed conversions belong in Silver.

**Job connection:** Idempotent batch loading allows a job to be retried after failure without duplicating previously received records. Stable lineage connects every database row back to its delivery and source position.

</details>


In [ ]:
from big_data_example.labs.retail_sales.wh_m01_bronze import (
    check_stage1,
    create_bronze_schema,
    load_bronze,
    read_bronze,
)

create_bronze_schema(project_root)
loaded_row_count = load_bronze(project_root)
print(f"Loaded Bronze rows: {loaded_row_count}")


### Checkpoint 1.5: Inspect Bronze with pandas and `psql`

<details>
<summary>Open Bronze object-and-shape explanation, SQL checks, and job connection</summary>

Input object: a SQL query over one Bronze batch. `pd.read_sql_query` returns `bronze_df`, a pandas DataFrame with shape `(40, 16)`. Selecting the eleven source columns returns `(40, 11)` and should exactly match `source_df`. All 40 source rows survive; zero rows are cleaned or rejected.

You can inspect the same state inside `psql` after connecting to `week3_medallion_lab`:

```text
\dt bronze.*
\d bronze.sales_transactions
SELECT COUNT(*) FROM bronze.sales_transactions;
SELECT source_row_number, transaction_id, transaction_date, quantity, unit_price, currency FROM bronze.sales_transactions ORDER BY source_row_number LIMIT 20;
```

**Hint:** `\d` describes table structure. `SELECT` is SQL and therefore ends with a semicolon; `psql` backslash commands do not.

**Job connection:** Inspecting both DataFrames and database catalogs separates in-memory assumptions from persisted reality. Bronze count and lineage checks are the first source-to-target reconciliation boundary.

</details>


In [ ]:
bronze_df = read_bronze(project_root)
print(f"Object type: {type(bronze_df).__name__}")
print(f"Shape: {bronze_df.shape}")
display(bronze_df.head(8))
display(bronze_df.dtypes.astype(str).rename("dtype").to_frame())

bronze_source_df = bronze_df.loc[:, list(SOURCE_COLUMNS)].astype("string")
source_values_preserved = bronze_source_df.equals(source_df)
print(f"Source values preserved exactly: {source_values_preserved}")


### Checkpoint 1.6: Check my work

<details>
<summary>Open validation instructions and job connection</summary>

Run the next cell. It verifies the deterministic files, checksum, exact 40-to-40 reconciliation, unique source row numbers, stable batch/file lineage, and byte-for-byte preservation of every source field.

**Job connection:** A successful insert proves that SQL ran. Reconciliation and preservation checks prove that the intended delivery reached Bronze without silent loss or mutation.

</details>


In [ ]:
stage1_evidence = check_stage1(project_root)
stage1_evidence


## Stage 1 stop point

Stop here and inspect Bronze before cleaning. Stage 1 is complete when the checker reports `PASS`, source and Bronze counts are both 40, Bronze has 16 columns, and you can explain why empty strings and invalid text remain unchanged.


## Stage 2: Bronze → Silver

<details>
<summary>Open stage plan, hint, and job connection</summary>

Clean with small staged pandas variables, preserve rejected records and understandable reasons, load typed Silver tables, and prove `Bronze rows = accepted rows + rejected rows` in a **Check my work** cell. Stop before Gold design.

**Hint:** Keep one Boolean validation column per rule before combining them. This makes it possible to explain exactly why each rejected row failed.

**Job connection:** A trustworthy pipeline does not silently discard invalid data. Quarantine plus reconciliation lets operators repair, replay, and account for every input record.

</details>


## Stage 3: Silver → Gold

<details>
<summary>Open stage plan, hint, and job connection</summary>

Declare the fact grain as one row per product line per transaction, build the star schema, resolve surrogate dimension keys, demonstrate SCD Type 1 and Type 2 behavior, and validate Gold with a **Check my work** cell.

**Hint:** State the fact grain before writing tables or joins. Every dimension lookup and uniqueness rule should preserve that declared grain.

**Job connection:** Clear grain and surrogate-key rules prevent double counting and allow facts to retain the historically correct dimension version when descriptive attributes change.

</details>


## Stage 4: Analytics and visualization

<details>
<summary>Open stage plan, hint, and job connection</summary>

Run beginner-friendly analytical SQL, load selected query results into pandas, and chart an already aggregated DataFrame with seaborn or Matplotlib.

**Hint:** Aggregate in PostgreSQL first, then inspect the small result's type and shape before passing it to the plotting function.

**Job connection:** Warehouses perform durable filtering, joining, and aggregation close to the data. Visualization tools should usually receive a bounded result rather than an entire raw fact table.

</details>


## Stage 5: Verification and reset

<details>
<summary>Open stage plan, hint, and job connection</summary>

Verify exact counts, fact grain, foreign keys, calculations, one current SCD2 row per customer, rejection reasons, top-to-bottom reruns, and each bounded cleanup path.

**Hint:** Treat successful execution as only the first check. Correctness requires assertions about the data that execution produced.

**Job connection:** Production teams need automated evidence and recovery instructions, not a notebook that happened to work once. Repeatable validation turns a demonstration into an operable pipeline.

</details>
